# SAMPLE — canonical longitudinal workflow (model-agnostic template)

**This notebook is a template, not an experiment, and is not meant to run as-is.**
Every model-specific operation is a placeholder `raise NotImplementedError` hook.
It documents the canonical end-to-end longitudinal pipeline — seed → data → CV →
threshold → save → test → ROC → early-detection → trajectories — shared by all
model notebooks (GELSTM, GEC-MLP, …).

The **SHARED** cells are byte-for-byte reusable across models. Everything
model-specific is funneled through the six hooks in the *Model adapter contract*
cell below. Those hook signatures double as the spec for the logic that will move
into `model/<family>/` scripts in the next refactor step.

Naming: the `BASELINE_/LONGITUDINAL_/STATIC_/SANITY_/COMPARISON_` prefix taxonomy
governs *experiment* notebooks; this template lives in `notebooks/SAMPLE/` with a
`SAMPLE_` prefix to mark its non-experiment status.

In [ ]:
# === Papermill parameters (injected by run_experiment.py) ===
# Safe interactive defaults: None keeps the original Jupyter behaviour
# (interactive checkpoint/threshold prompts, JSON-config loading).
EXPERIMENT_ID = None
MODE = None
MODEL = None
DATASET = None
SEED = 42
GAAE_CHECKPOINT_PATH = None   # None -> interactive checkpoint picker
THRESHOLD_MODE = None         # None -> interactive prompt; else youden | best-f1 | fixed
FIXED_THRESHOLD = None        # required when THRESHOLD_MODE is fixed
WANDB_ENABLED = True          # W&B logging is on by default
OUTPUT_DIR = None             # defaults to a sample checkpoints dir when run standalone
RESOLVED_CONFIG = None        # merged hyperparameter dict; overrides on-disk JSON when set
RUN_DIR = None                # set by the runner: where run_summary.json / artifacts go
RUN_NAME = None               # set by the runner: the W&B run name

## Pipeline overview

`set_seed` → load splits → `prepare_data` (HOOK) → `build_model` (HOOK) →
5-fold StratifiedGroupKFold CV (`train_fold` HOOK) → threshold selection →
save full-state checkpoint → test-set evaluation (`eval_split` HOOK) →
ROC curves → early-detection curve (`truncate_to_n_visits` HOOK) →
conversion trajectories (`per_visit_probs` HOOK).

In [ ]:
import sys
from pathlib import Path
repo_root = Path('/mnt/e/fyassine/ad-early-detection')
model_root = Path('/mnt/e/fyassine/ad-early-detection/CLASSIFIER')
if str(model_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
    sys.path.insert(0, str(model_root))

In [ ]:
# v2 reproducibility seeding — must run before datasets, samplers, or models.
from CLASSIFIER.common.seeding import (
    set_seed, make_rng, make_torch_generator, seed_worker,
)
set_seed(SEED)
rng = make_rng(SEED)
torch_gen = make_torch_generator(SEED)

In [ ]:
import json, os, copy, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from datetime import datetime
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    roc_auc_score, roc_curve, f1_score, confusion_matrix, classification_report,
)

# Shared, model-agnostic utilities (these already live in scripts — reuse them).
from common.checkpoints import select_gaae_checkpoint
from common.sanity import run_full_audit
from common import tracking
from common.provenance import (
    region_from_data_root, make_run_dir, snapshot_source,
    save_full_checkpoint, write_run_summary, patch_run_summary,
    capture_env, capture_git_provenance,
)

# ── MODEL-SPECIFIC IMPORTS (fill in per model notebook) ───────────────────
# e.g. GELSTM:
#   from model.GELSTM.models  import GELSTMClassifier
#   from model.GELSTM.dataset import LongitudinalSubjectDataset
#   from model.GELSTM.train   import train_model, evaluate, make_batches
#   from model.GELSTM.utils   import encode_batch_sequences, compute_class_weights
# e.g. GEC (flat MLP):
#   from model.GAAE.models   import GraphAttentionAutoencoderConditioned
#   from model.GELSTM.dataset import LongitudinalSubjectDataset

warnings.filterwarnings('ignore')
print('Imports OK')

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## Model adapter contract

A concrete model notebook implements the six hooks below (replacing each
`raise NotImplementedError`). The downstream SHARED cells call only these hooks
and a `Bundle` container, so they remain identical across models:

| Hook | Responsibility |
| --- | --- |
| `build_model()` | instantiate model, load + freeze GAAE encoder |
| `prepare_data(df)` | encode a split into a `Bundle` |
| `train_fold(tr, va, cfg)` | train + select on one CV fold |
| `eval_split(state, bundle, thr)` | score a split at a fixed threshold |
| `truncate_to_n_visits(bundle, n)` | first-N-visits view (early detection) |
| `per_visit_probs(state, item)` | per-visit prefix probabilities (trajectories) |

In [ ]:
# === MODEL-SPECIFIC HOOKS ====================================================
# This template is model-agnostic: every model-specific operation is funneled
# through the functions below. A concrete model notebook replaces each
# `raise NotImplementedError` body; the SHARED cells downstream call ONLY these
# hooks (and the Bundle container), so they stay identical across models.
#
# These signatures are the spec for what will move into `model/<family>/`
# scripts in the next step of the refactor.

class Bundle:
    """Model-agnostic container for one split's encoded data.

    A concrete model notebook supplies its own Bundle (or any object exposing
    these members). The SHARED cells rely only on this interface:

      .labels : list[int]   per-subject labels {0,1}, aligned with .items
      .groups : list[str]   per-subject ids (StratifiedGroupKFold groups)
      .items  : list[dict]  per-subject records (trajectories / visit truncation);
                            each item exposes at least 'subject_id','label','n_scans'.
      .subset(idx) -> Bundle  new Bundle restricted to positions idx.
    """
    def __init__(self, labels, groups, items):
        self.labels, self.groups, self.items = labels, groups, items

    def subset(self, idx):
        idx = list(idx)
        return Bundle([self.labels[i] for i in idx],
                      [self.groups[i] for i in idx],
                      [self.items[i]  for i in idx])


def build_model() -> 'nn.Module':
    """Instantiate the model (load + freeze the GAAE encoder), print param counts."""
    raise NotImplementedError('per-model hook: build and return the model')


def prepare_data(df: 'pd.DataFrame') -> Bundle:
    """Encode a split DataFrame into a model-specific Bundle.

    GELSTM: wrap LongitudinalSubjectDataset items directly.
    GEC:    encode each visit through the frozen GAAE, flatten to fixed vectors,
            and carry (X, y, groups) alongside the per-subject records.
    """
    raise NotImplementedError('per-model hook: encode df into a Bundle')


def train_fold(bundle_tr: Bundle, bundle_va: Bundle, cfg: dict, *, rng, device) -> dict:
    """Train one CV fold. Returns:
        {
          'state_dict':     best model state (val-AUC selected),
          'val_metrics':    {'auc','sensitivity','specificity','f1'},
          'best_threshold': float (val-derived),
          'oof_probs':      np.ndarray, 'oof_targets': np.ndarray, 'oof_sids': list[str],
        }
    """
    raise NotImplementedError('per-model hook: train + select on the val fold')


def eval_split(state_dict, bundle: Bundle, threshold: float, *, device) -> dict:
    """Evaluate a trained model on a bundle at a FIXED threshold (never re-optimised
    on the data being scored — see .claude/rules/evaluation.md). Returns:
        {'auc','sensitivity','specificity','f1','probs','targets','subject_ids','n_scans'}
    """
    raise NotImplementedError('per-model hook: score bundle at the given threshold')


def truncate_to_n_visits(bundle: Bundle, n: int) -> Bundle:
    """Return a Bundle with each subject restricted to their first `n` visits
    (subjects with fewer than `n` visits are dropped). Used by the early-detection curve.
    """
    raise NotImplementedError('per-model hook: first-N-visits truncation')


def per_visit_probs(state_dict, item: dict, *, device) -> list:
    """Return [(visit_month, P(converter)), ...] for prefix sequences of length 1..T
    of a single subject `item`. Used by the trajectory plot.
    """
    raise NotImplementedError('per-model hook: per-visit prefix probabilities')

## Configuration

In [ ]:
from DATA.src.splitting.load_splits import splits_dir, split_csv_paths

# ── Paths ────────────────────────────────────────────────────────────────
WB_DATA_ROOT = '/mnt/e/fyassine/ad-early-detection/DATA/DELCODE/__fc_wholebrain_sch200_flat__/matrices'
METADATA_DIR = '/mnt/e/fyassine/ad-early-detection/DATA/DELCODE/__fc_wholebrain_sch200_flat__/metadata'
COHORTS_CSV  = os.path.join(METADATA_DIR, 'cohorts.csv')
SPLITS_DIR   = str(splits_dir('downstream'))
TRAIN_CSV    = os.path.join(SPLITS_DIR, 'train.csv')
VAL_CSV      = os.path.join(SPLITS_DIR, 'val.csv')
TEST_CSV     = os.path.join(SPLITS_DIR, 'test.csv')

# Brain region / atlas parsed from the DATA directory name (surfaced in run name + config).
DATA_INFO = region_from_data_root(WB_DATA_ROOT)
REGION    = DATA_INFO['region']
print(f"Input data: region={DATA_INFO['region']}  atlas={DATA_INFO['atlas']}  ({DATA_INFO['dataset_dir']})")

CHECKPOINT_SEARCH_DIRS = [
    str(model_root / 'notebooks' / 'checkpoints' / 'checkpoints_gaae_whole_brain'),
]
if OUTPUT_DIR is None:
    OUTPUT_DIR = str(model_root / 'notebooks' / 'checkpoints' / 'checkpoints_sample_template')
os.makedirs(OUTPUT_DIR, exist_ok=True)

# GAAE encoder config (must match checkpoint).
CONFIG_PATH = model_root / 'configs' / 'gaae_delcode_whole_brain.json'

# ── MODEL-SPECIFIC: training hyper-params ─────────────────────────────────
# Point TRAIN_CONFIG_PATH at this model's configs/*.json. The load + RESOLVED_CONFIG
# merge below is shared boilerplate; only the path and the hyperparam unpacking change.
TRAIN_CONFIG_PATH = model_root / 'configs' / '<model>_delcode_whole_brain.json'   # EDIT per model
_train_defaults = {'epochs': 50, 'batch_size': 16, 'learning_rate': 1e-3,
                   'grad_clip': 1.0, 'early_stopping_patience': 15, 'n_folds': 5}
if TRAIN_CONFIG_PATH.exists():
    with open(TRAIN_CONFIG_PATH) as _f:
        TRAIN_CONFIG = json.load(_f)
    print(f'Loaded training config from {TRAIN_CONFIG_PATH}')
else:
    TRAIN_CONFIG = _train_defaults
    print(f'Training config not found at {TRAIN_CONFIG_PATH} — using inline defaults.')

# Runner override: merge injected RESOLVED_CONFIG (YAML hyperparams) over JSON config.
if RESOLVED_CONFIG:
    TRAIN_CONFIG = {**TRAIN_CONFIG, **RESOLVED_CONFIG}
    print('Applied RESOLVED_CONFIG overrides from runner.')

N_FOLDS = TRAIN_CONFIG.get('n_folds', 5)
print('Config set.')

In [ ]:
# v2 split-hygiene audit — hard-fails if any subject crosses splits.
_ = run_full_audit(split_csv_paths('downstream'))

## Select GAAE Checkpoint

In [ ]:
# Shared helper (already in common/checkpoints.py): lists candidates and prompts for
# an index interactively, or resolves GAAE_CHECKPOINT_PATH headlessly under the runner.
if GAAE_CHECKPOINT_PATH is None and RUN_DIR is not None:
    raise ValueError(
        "GAAE_CHECKPOINT_PATH is required under the experiment runner. "
        "Set 'checkpoint_path:' on this entry in experiments.yaml."
    )
GAAE_RUN_NAME, GAAE_CKPT_PATH, GAAE_RUN_DIR = select_gaae_checkpoint(
    CHECKPOINT_SEARCH_DIRS, checkpoint_path=GAAE_CHECKPOINT_PATH,
)
print(f'Selected GAAE: {GAAE_RUN_NAME}')

In [ ]:
if CONFIG_PATH.exists():
    with open(CONFIG_PATH) as f: hp = json.load(f)
    print('GAAE config:', hp)
else:
    hp = dict(latent_dim=64, hidden_dim=128, num_heads=2, cond_dim=2, dropout=0.3,
              adjacency_k=8, file_variant='z_transformed')
    print('GAAE config not found — using defaults.')

IN_FEATURES   = 200   # Schaefer-200 ROIs
GAAE_HIDDEN   = hp.get('hidden_dim', 128)
GAAE_LATENT   = hp.get('latent_dim', 64)
GAAE_HEADS    = hp.get('num_heads', 2)
GAAE_COND_DIM = hp.get('cond_dim', 2)
GAAE_DROPOUT  = hp.get('dropout', 0.3)
ADJ_K         = hp.get('adjacency_k', 8)
FILE_VARIANT  = hp.get('file_variant', 'z_transformed')
print(f'GAAE hidden={GAAE_HIDDEN}  latent={GAAE_LATENT}  heads={GAAE_HEADS}')

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
val_df   = pd.read_csv(VAL_CSV)
test_df  = pd.read_csv(TEST_CSV)

# CV pool = train + val; test held out.
cv_pool_df = pd.concat([train_df, val_df], ignore_index=True)

print('CV pool:', cv_pool_df['diagnosis'].value_counts().to_dict())
print('Test:   ', test_df['diagnosis'].value_counts().to_dict())

In [ ]:
# HOOK calls: encode each split into a model-specific Bundle, then smoke-test the model.
cv_bundle   = prepare_data(cv_pool_df)
test_bundle = prepare_data(test_df)
print(f'CV subjects: {len(cv_bundle.items)}  Test subjects: {len(test_bundle.items)}')

_ = build_model()   # smoke-test arch construction

## 5-Fold Stratified Subject-Level Cross-Validation

In [ ]:
cv_results = {'fold':[],'val_auc':[],'val_sensitivity':[],'val_specificity':[],'val_f1':[],'best_threshold':[]}
oof_probs, oof_targets, oof_sids = [], [], []

best_val_auc, best_fold, best_model_state = 0.0, -1, None
best_threshold_overall = 0.5

_wb_exp = {'id': EXPERIMENT_ID or 'sample-template', 'mode': MODE or 'longitudinal',
           'model': MODEL or 'SAMPLE', 'dataset': DATASET or REGION,
           'seed': SEED, 'wandb': WANDB_ENABLED}
wandb_run = tracking.init_run(_wb_exp, {**(RESOLVED_CONFIG or {}), 'REGION': REGION})

# NOTE (refactor): this StratifiedGroupKFold loop is identical across models and is the
# prime candidate to lift into a shared common/crossval.py helper next. (The rules mention
# common.validation.run_kfold_cv, but that module was deleted — no shared helper exists yet.)
sgkf = StratifiedGroupKFold(n_splits=N_FOLDS)
for fold, (tr_idx, va_idx) in enumerate(sgkf.split(cv_bundle.items, cv_bundle.labels, groups=cv_bundle.groups)):
    print('=' * 55)
    print(f'Fold {fold+1}/{N_FOLDS}  train={len(tr_idx)}  val={len(va_idx)}')

    fold_out = train_fold(cv_bundle.subset(tr_idx), cv_bundle.subset(va_idx),
                          TRAIN_CONFIG, rng=rng, device=device)

    vm = fold_out['val_metrics']
    oof_probs.extend(list(fold_out['oof_probs']))
    oof_targets.extend(list(fold_out['oof_targets']))
    oof_sids.extend(list(fold_out['oof_sids']))

    cv_results['fold'].append(fold + 1)
    cv_results['val_auc'].append(vm['auc'])
    cv_results['val_sensitivity'].append(vm['sensitivity'])
    cv_results['val_specificity'].append(vm['specificity'])
    cv_results['val_f1'].append(vm['f1'])
    cv_results['best_threshold'].append(fold_out['best_threshold'])
    tracking.log_metrics(wandb_run, {'fold': fold+1, 'val_auc': vm['auc'], 'val_f1': vm['f1']})

    print(f"  AUC={vm['auc']:.4f}  sens={vm['sensitivity']:.3f}  "
          f"spec={vm['specificity']:.3f}  F1={vm['f1']:.3f}")

    if vm['auc'] > best_val_auc:
        best_val_auc, best_fold = vm['auc'], fold + 1
        best_model_state = fold_out['state_dict']
        best_threshold_overall = fold_out['best_threshold']

oof_arr = np.array(oof_probs); oof_tgt = np.array(oof_targets)
_, _, thrs = roc_curve(oof_tgt, oof_arr)
best_f1_thr = float(thrs[np.argmax([f1_score(oof_tgt, (oof_arr>=t).astype(int), zero_division=0) for t in thrs])])
print(f'\nBest fold: {best_fold}  CV AUC={best_val_auc:.4f}')
print(f'Youden thr={best_threshold_overall:.4f}  OOF-F1 thr={best_f1_thr:.4f}')

## Cross-Validation Summary

In [ ]:
print('Cross-Validation Summary:')
print('=' * 60)
print(f"{'Metric':<20} {'Mean':>10} {'Std':>10} {'Min':>10} {'Max':>10}")
print('-' * 60)
for m in ['val_auc','val_sensitivity','val_specificity','val_f1']:
    v = cv_results[m]
    print(f'{m:<20} {np.mean(v):>10.4f} {np.std(v):>10.4f} {np.min(v):>10.4f} {np.max(v):>10.4f}')

In [ ]:
def _oof_metrics(thr):
    p = (oof_arr >= thr).astype(int)
    if len(np.unique(oof_tgt)) > 1:
        tn, fp, fn, tp = confusion_matrix(oof_tgt, p).ravel()
    else:
        tn = fp = fn = tp = 0
    sens = tp/(tp+fn) if (tp+fn) > 0 else 0.0
    spec = tn/(tn+fp) if (tn+fp) > 0 else 0.0
    return sens, spec, f1_score(oof_tgt, p, zero_division=0)

f_s, f_sp, f_f1 = _oof_metrics(best_f1_thr)
y_s, y_sp, y_f1 = _oof_metrics(best_threshold_overall)
print('OOF threshold options:')
print(f'  [1] Best-F1 (default) thr={best_f1_thr:.4f}  sens={f_s:.3f}  spec={f_sp:.3f}  F1={f_f1:.3f}')
print(f'  [2] Youden            thr={best_threshold_overall:.4f}  sens={y_s:.3f}  spec={y_sp:.3f}  F1={y_f1:.3f}')

# Best-F1 is the default (option 1 / Enter) per .claude/rules/notebooks.md.
if THRESHOLD_MODE == 'best-f1':
    ACTIVE_THRESHOLD = best_f1_thr; THRESHOLD_METHOD = 'oof_f1'
elif THRESHOLD_MODE == 'youden':
    ACTIVE_THRESHOLD = best_threshold_overall; THRESHOLD_METHOD = 'oof_youden'
elif THRESHOLD_MODE == 'fixed':
    if FIXED_THRESHOLD is None:
        raise ValueError("THRESHOLD_MODE='fixed' requires FIXED_THRESHOLD")
    ACTIVE_THRESHOLD = float(FIXED_THRESHOLD); THRESHOLD_METHOD = 'fixed'
elif RUN_DIR is not None:
    raise ValueError(
        "THRESHOLD_MODE is required under the experiment runner "
        "(youden | best-f1 | fixed). Set 'threshold_mode:' in experiments.yaml."
    )
else:
    choice = input('Select threshold [1=Best-F1 (default), 2=Youden]: ').strip()
    if choice == '2':
        ACTIVE_THRESHOLD = best_threshold_overall; THRESHOLD_METHOD = 'oof_youden'
    else:
        ACTIVE_THRESHOLD = best_f1_thr; THRESHOLD_METHOD = 'oof_f1'
print(f'Using {THRESHOLD_METHOD} threshold: {ACTIVE_THRESHOLD:.4f}')

## Save Best Model

In [ ]:
# Create the run directory (region embedded in run name) and save artifacts.
if RUN_DIR:
    run_dir = Path(RUN_DIR); run_dir.mkdir(parents=True, exist_ok=True)
    run_name = RUN_NAME or run_dir.name
else:
    run_name, run_dir = make_run_dir(OUTPUT_DIR, 'sample', DATA_INFO)

# Back-compat artifact: plain state_dict (loaded directly by comparison / dashboard code).
torch.save(best_model_state, run_dir / f'model_{run_name}.pth')

# ── MODEL-SPECIFIC: describe the architecture so the run can be rebuilt ─────
model_config = {
    'model_type': '<ModelClass>',           # EDIT per model
    'in_features': IN_FEATURES,
    'gaae_hidden': GAAE_HIDDEN, 'gaae_latent': GAAE_LATENT,
    'gaae_heads': GAAE_HEADS, 'gaae_cond_dim': GAAE_COND_DIM, 'gaae_dropout': GAAE_DROPOUT,
    # ... add this model's own arch fields (lstm_hidden / mlp_hidden_layers / ...)
}
dataset_info = {**DATA_INFO, 'train_csv': TRAIN_CSV, 'val_csv': VAL_CSV,
                'test_csv': TEST_CSV, 'n_folds': N_FOLDS}

# Full-state checkpoint (state + RNG + config) for flawless reload + rerun.
save_full_checkpoint(
    run_dir / f'checkpoint_{run_name}.pth',
    model_state=best_model_state,
    model_config=model_config,
    training_config=TRAIN_CONFIG,
    rng=rng,
    val_auc=float(best_val_auc),
    best_threshold=float(ACTIVE_THRESHOLD),
    threshold_method=THRESHOLD_METHOD,
    best_fold=int(best_fold),
    gaae_checkpoint=GAAE_CKPT_PATH,
    run_name=run_name,
)

# Snapshot the source that produced this run ("save code") + git commit.
# ── MODEL-SPECIFIC: list this model's source files ─────────────────────────
snapshot_source(run_dir, [
    model_root / 'model' / 'GAAE' / 'models.py',
    # model_root / 'model' / '<FAMILY>' / 'models.py',
    # model_root / 'model' / '<FAMILY>' / 'train.py',
])

run_summary = {
    'run_name': run_name, 'data_info': DATA_INFO, 'dataset_info': dataset_info,
    'model_config': model_config, 'training_config': TRAIN_CONFIG,
    'gaae_checkpoint': GAAE_CKPT_PATH, 'gaae_run_name': GAAE_RUN_NAME,
    'n_folds': N_FOLDS, 'best_fold': int(best_fold),
    'best_val_auc': float(best_val_auc),
    'active_threshold': float(ACTIVE_THRESHOLD), 'threshold_method': THRESHOLD_METHOD,
    'cv_results': cv_results,
    'env': capture_env(), 'git': capture_git_provenance(),
}
write_run_summary(run_dir, run_summary)
print(f'Saved to {run_dir}')

try:
    tracking.log_metrics(wandb_run, {'cv_best_val_auc': float(best_val_auc),
                                     'active_threshold': float(ACTIVE_THRESHOLD)})
except NameError:
    pass

## Test-Set Evaluation

In [ ]:
# HOOK: score the held-out test set at the val-derived threshold (no test leakage).
te = eval_split(best_model_state, test_bundle, ACTIVE_THRESHOLD, device=device)

print('Test-Set Results')
print('=' * 40)
print(f"AUC:         {te['auc']:.4f}")
print(f"Sensitivity: {te['sensitivity']:.4f}")
print(f"Specificity: {te['specificity']:.4f}")
print(f"F1:          {te['f1']:.4f}")
print(f"Threshold:   {ACTIVE_THRESHOLD:.4f}  ({THRESHOLD_METHOD})")
print()
print(classification_report(te['targets'], (np.asarray(te['probs']) >= ACTIVE_THRESHOLD).astype(int),
                            target_names=['stable_mci', 'converter']))

# Back-patch run_summary.json with test metrics.
patch_run_summary(run_dir, {
    'metrics': {
        'test_auc': float(te['auc']), 'test_f1': float(te['f1']),
        'test_sensitivity': float(te['sensitivity']), 'test_specificity': float(te['specificity']),
        'threshold': float(ACTIVE_THRESHOLD), 'threshold_method': THRESHOLD_METHOD,
    },
    'test_auc': float(te['auc']), 'test_sensitivity': float(te['sensitivity']),
    'test_specificity': float(te['specificity']), 'test_f1': float(te['f1']),
    'test_probabilities': [float(p) for p in te['probs']],
    'test_labels': [int(t) for t in te['targets']],
})
print(f"Test metrics saved to {run_dir / 'run_summary.json'}")

try:
    tracking.log_metrics(wandb_run, {'test_auc': float(te['auc']), 'test_f1': float(te['f1']),
                                     'test_sensitivity': float(te['sensitivity']),
                                     'test_specificity': float(te['specificity'])})
    tracking.finish_run(wandb_run)
except NameError:
    pass

## ROC Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: OOF ROC
ax = axes[0]
fpr_oof, tpr_oof, _ = roc_curve(oof_tgt, oof_arr)
auc_oof = roc_auc_score(oof_tgt, oof_arr)
ax.plot(fpr_oof, tpr_oof, lw=2, color='#2196F3', label=f'OOF ROC (AUC={auc_oof:.3f})')
ax.plot([0, 1], [0, 1], '--', color='grey', alpha=0.5)
ax.set_xlabel('FPR'); ax.set_ylabel('TPR'); ax.set_title('OOF Cross-Validation ROC')
ax.legend(); ax.grid(alpha=0.3)

# Right: test ROC
ax = axes[1]
fpr_te, tpr_te, _ = roc_curve(np.asarray(te['targets']).astype(int), te['probs'])
ax.plot(fpr_te, tpr_te, lw=2, color='#F44336', label=f"Test ROC (AUC={te['auc']:.3f})")
ax.plot([0, 1], [0, 1], '--', color='grey', alpha=0.5)
ax.set_xlabel('FPR'); ax.set_ylabel('TPR'); ax.set_title('Test-Set ROC')
ax.legend(); ax.grid(alpha=0.3)

fig.suptitle(f'SAMPLE template  |  GAAE: {GAAE_RUN_NAME}', fontsize=10)
plt.tight_layout(); plt.show()

## Early-Detection Curve: AUC vs. Number of Visits Used

For each `N`, take all test subjects with `T >= N` visits, restrict each to their
first `N` visits (HOOK `truncate_to_n_visits`), and re-score with the trained model.
Rows with fewer than 4 subjects or a single class are skipped (same guard as the
GEC / GELSTM source notebooks). `N=1` = single-scan / baseline-only classification.

In [ ]:
MAX_SCANS = max(item['n_scans'] for item in test_bundle.items)
print(f'\n{"Visits":>6} {"N":>4} {"AUC":>8} {"Sens":>8} {"Spec":>8}')
print('-' * 40)
for n_vis in range(1, MAX_SCANS + 1):
    sub_bundle = truncate_to_n_visits(test_bundle, n_vis)
    if len(sub_bundle.items) < 4 or len(np.unique(sub_bundle.labels)) < 2:
        continue
    m = eval_split(best_model_state, sub_bundle, ACTIVE_THRESHOLD, device=device)
    print(f"{n_vis:>6} {len(sub_bundle.items):>4} {m['auc']:>8.4f} "
          f"{m['sensitivity']:>8.3f} {m['specificity']:>8.3f}")

## Conversion Probability Trajectories

For each test subject with >= 2 visits, compute `P(converter)` for prefix sequences
of length 1..T (HOOK `per_visit_probs`) and plot the trajectory, split by outcome.

In [ ]:
traj_records = []
for item in test_bundle.items:
    if item['n_scans'] < 2:
        continue
    for month, prob in per_visit_probs(best_model_state, item, device=device):
        traj_records.append({'pid': item['subject_id'], 'label': item['label'],
                             'month': month, 'prob': prob})
traj_df = pd.DataFrame(traj_records)
print(f"Trajectory data: {traj_df['pid'].nunique()} subjects")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
palette = {1: '#F44336', 0: '#2196F3'}
for ax, (label, title) in zip(axes, [(1, 'Converters'), (0, 'Stable MCI')]):
    sub = traj_df[traj_df['label'] == label]
    for pid, grp in sub.groupby('pid'):
        ax.plot(grp['month'], grp['prob'], marker='o', alpha=0.6, color=palette[label], lw=1.5)
    ax.axhline(ACTIVE_THRESHOLD, color='black', lw=1.2, linestyle='--',
               label=f'Threshold={ACTIVE_THRESHOLD:.3f}')
    ax.set_xlabel('Visit month'); ax.set_ylabel('P(converter)')
    ax.set_ylim(-0.05, 1.05); ax.set_title(title); ax.legend(fontsize=8); ax.grid(alpha=0.3)
fig.suptitle('Per-visit conversion probability trajectories  |  SAMPLE template', fontsize=11)
plt.tight_layout(); plt.show()